# Módulo 13 — Projeto: Fundamentos da Descoberta de Dados

**Objetivo:** analisar os preços normais e descontos dos produtos de supermercado por categoria e marca, construir visualizações e interpretar os resultados.

A base contém 1.107 produtos, 7 categorias e 156 marcas. O arquivo CSV deve estar na mesma pasta deste notebook.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

# Carregamento: o CSV fornecido usa vírgula como separador.
df = pd.read_csv("MODULO7_PROJETOFINAL_BASE_SUPERMERCADO.csv")

# Conferência inicial
display(df.head())
print("Dimensões:", df.shape)
print("\\nValores ausentes por coluna:")
display(df.isna().sum())
print("\\nTipos de dados:")
print(df.dtypes)

## 1. Média e mediana de Preco_Normal por categoria

A comparação é feita entre a média e a mediana de cada categoria. Média acima da mediana pode indicar assimetria à direita, com produtos mais caros elevando a média.

In [ ]:
estatisticas = df.groupby("Categoria")["Preco_Normal"].agg(
    media="mean",
    mediana="median"
).sort_values("media", ascending=False)

estatisticas["comparacao"] = estatisticas.apply(
    lambda linha: "Média acima da mediana" if linha["media"] > linha["mediana"]
    else ("Média abaixo da mediana" if linha["media"] < linha["mediana"]
          else "Média igual à mediana"),
    axis=1
)
display(estatisticas.round(2))

**Interpretação:** a média está acima da mediana em `lacteos`, `belleza-y-cuidado-personal`, `congelados`, `frutas`, `instantaneos-y-sopas` e `verduras`. Em `comidas-preparadas`, a média está abaixo da mediana. Esse padrão sugere que, na maioria das categorias, alguns produtos de preço mais alto elevam a média.

## 2. Desvio padrão por categoria

O desvio padrão mede a dispersão dos preços em torno da média. Quanto maior o valor, maior a variabilidade observada.

In [ ]:
desvios = df.groupby("Categoria")["Preco_Normal"].std().sort_values(ascending=False)
display(desvios.to_frame("desvio_padrao").round(2))

categoria_maior_desvio = desvios.idxmax()
print("Categoria com maior desvio padrão:", categoria_maior_desvio)

**Interpretação:** a categoria `lacteos` apresenta o maior desvio padrão (aproximadamente 3.925,82). Nessa categoria, a média (2.385,22) é bem superior à mediana (989,00), sugerindo assimetria à direita e influência de produtos com preços bastante elevados.

## 3. Boxplot e identificação de outliers

Os outliers são identificados pelo critério do intervalo interquartil (IQR): valores abaixo de Q1 − 1,5 × IQR ou acima de Q3 + 1,5 × IQR.

In [ ]:
dados_lacteos = df.loc[df["Categoria"] == categoria_maior_desvio, "Preco_Normal"]
q1 = dados_lacteos.quantile(0.25)
q3 = dados_lacteos.quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
outliers = dados_lacteos[
    (dados_lacteos < limite_inferior) | (dados_lacteos > limite_superior)
]

plt.figure(figsize=(9, 5))
plt.boxplot(dados_lacteos, showmeans=True)
plt.title("Distribuição de Preco_Normal — Lácteos")
plt.ylabel("Preço normal (CLP)")
plt.xticks([1], ["Lácteos"])
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig("m13_boxplot_lacteos.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"Q1: {q1:.2f} | Q3: {q3:.2f} | IQR: {iqr:.2f}")
print(f"Limite inferior: {limite_inferior:.2f} | Limite superior: {limite_superior:.2f}")
print(f"Quantidade de outliers: {len(outliers)} de {len(dados_lacteos)} produtos")
display(outliers.sort_values(ascending=False).to_frame("Preco_Normal"))

**Interpretação:** o boxplot evidencia uma distribuição assimétrica à direita, com vários valores acima do limite superior. Pelo critério de 1,5 × IQR, foram identificados **43 outliers** entre 447 produtos lácteos. O limite superior calculado é 6.119 CLP; os valores acima dele são considerados outliers. Esses registros podem representar produtos de maior valor ou características específicas do catálogo, não necessariamente erros.

## 4. Gráfico de barras — desconto médio por categoria

A média é calculada incluindo os registros com desconto igual a zero, conforme a estrutura da base.

In [ ]:
desconto_categoria = df.groupby("Categoria")["Desconto"].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
ax = desconto_categoria.plot(kind="bar")
ax.set_title("Desconto médio por categoria")
ax.set_xlabel("Categoria")
ax.set_ylabel("Desconto médio (CLP)")
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig("m13_barras_desconto_categoria.png", dpi=200, bbox_inches="tight")
plt.show()

display(desconto_categoria.round(2).to_frame("desconto_medio"))

**Interpretação:** congelados apresenta o maior desconto médio (154,03 CLP), seguido por beleza e cuidados pessoais (123,08 CLP). Frutas, instantâneos e sopas e verduras apresentam média zero, pois não há descontos registrados nesses grupos na coluna `Desconto`.

## 5. Mapa interativo (treemap) — categoria, marca e desconto médio

O treemap agrupa os produtos hierarquicamente por categoria e marca. A área representa a quantidade de produtos em cada combinação, enquanto a cor representa o desconto médio. Ao passar o cursor sobre os blocos, é possível consultar a média e a quantidade.

In [ ]:
agrupado = df.groupby(["Categoria", "Marca"], as_index=False).agg(
    Desconto_Medio=("Desconto", "mean"),
    Quantidade_Produtos=("Desconto", "size")
)

fig = px.treemap(
    agrupado,
    path=["Categoria", "Marca"],
    values="Quantidade_Produtos",
    color="Desconto_Medio",
    color_continuous_scale="Viridis",
    hover_data={
        "Desconto_Medio": ":.2f",
        "Quantidade_Produtos": True
    },
    title="Desconto médio por categoria e marca"
)
fig.update_layout(
    margin=dict(t=55, l=10, r=10, b=10),
    coloraxis_colorbar_title="Desconto médio (CLP)"
)
fig.write_html("m13_mapa_interativo_treemap.html")
fig.show()

**Interpretação:** o treemap permite explorar simultaneamente a composição da base por categoria e marca. Blocos maiores representam marcas com maior quantidade de produtos na categoria; a escala de cores permite comparar o desconto médio entre essas combinações. O gráfico é interativo e complementa o gráfico de barras, que resume os descontos apenas por categoria.

## 6. Conclusão

A análise identificou grande variabilidade de preços na categoria de lácteos, que apresentou o maior desvio padrão e 43 outliers pelo critério do intervalo interquartil. Em seis das sete categorias, a média de preço normal ficou acima da mediana, indicando possível assimetria à direita. No estudo dos descontos, congelados apresentou a maior média. O treemap complementa a análise ao permitir investigar os descontos por categoria e marca.

**Limitação:** os resultados descrevem a base fornecida e não devem ser generalizados para outros supermercados ou períodos sem validação adicional.